<a href="https://colab.research.google.com/github/Longhanhmid24/DoAn_Email/blob/main/Qwen_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
base_path = "/content/drive/MyDrive/Deep_Learning_Email/dataset_split/"
train_df = pd.read_csv(base_path + "train.csv")
val_df   = pd.read_csv(base_path + "val.csv")
test_df  = pd.read_csv(base_path + "test.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
pip install -q -U transformers datasets evaluate accelerate bitsandbytes peft trl

In [3]:
!pip uninstall -y pyarrow
!pip install pyarrow

Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)


In [4]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from tqdm import tqdm

#### HUẤN LUYỆN QWEN (QLoRA FINE-TUNING)

In [5]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Cấu hình lượng tử hóa 4-bit để tiết kiệm bộ nhớ (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Qwen cần set pad_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# Cấu hình LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
from datasets import Dataset

# Qwen thường sử dụng định dạng ChatML (với các thẻ <|im_start|> và <|im_end|>)
def format_chatml(example):
    # Khác với T5, Qwen (Causal LM) cần gộp cả Input và Output thành 1 chuỗi liên tục để học
    prompt = f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
    prompt += f"<|im_start|>user\n{example['Input_Text']}<|im_end|>\n"
    prompt += f"<|im_start|>assistant\n{example['Output_Text']}<|im_end|>"
    return {"text": prompt}

In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.map(format_chatml)
val_dataset = val_dataset.map(format_chatml)

Map:   0%|          | 0/5116 [00:00<?, ? examples/s]

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

In [9]:
from trl import SFTConfig

# Cấu hình Trainer
training_args = SFTConfig(
    output_dir="./qwen_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,   # giảm mạnh
    max_grad_norm=1.0,    # chống nổ
    logging_steps=10,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
     gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Bắt đầu huấn luyện Qwen...")
trainer.train()

print("Đang lưu model...")

# 1. Lưu model + tokenizer
save_path = "/content/qwen_finetuned_final"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# 2. Nén lại (để tải nhanh hơn)
import shutil
shutil.make_archive(save_path, 'zip', save_path)

# 3. Download về máy
from google.colab import files
files.download(save_path + ".zip")

print(" Đã lưu và tải model về máy!")

Adding EOS to train dataset:   0%|          | 0/5116 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5116 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/639 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/639 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Đang lưu model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã lưu và tải model về máy!


#### ĐÁNH GIÁ VÀ VẼ BIỂU ĐỒ

In [11]:
!pip install -q rouge_score sacrebleu bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 10.9 MB/s eta 0:00:00


In [ ]:
import importlib
import evaluate
importlib.reload(evaluate)

print("Bắt đầu quá trình đánh giá trên tập Test...")

# Tải các bộ đánh giá
rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')
bertscore = evaluate.load('bertscore')

# Hàm sinh văn bản để đánh giá
def generate_responses(test_data, batch_size=64):
    model.eval()
    predictions = []
    references = []

    # Để tránh OOM, ta chạy từng mẫu (hoặc batch nhỏ)
    for example in tqdm(test_data, desc="Generating"):
        # Chỉ lấy phần câu hỏi để mô hình tự sinh phần trả lời
        prompt = f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n{example['Input_Text']}<|im_end|>\n<|im_start|>assistant\n"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                pad_token_id=tokenizer.eos_token_id,
                temperature=0.7,
                top_p=0.9
            )

        # Giải mã chuỗi sinh ra (Cắt bỏ phần Prompt ban đầu)
        generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        predictions.append(generated_text.strip())
        references.append(example['Output_Text'].strip())

    return predictions, references

predictions, references = generate_responses(test_dataset)

# 1. Tính toán các Metrics văn bản
rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])
bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")

bert_f1 = np.mean(bertscore_results['f1'])

# 2. Tính toán Perplexity (Sử dụng hàm evaluate của SFTTrainer trên test_dataset đã format)
# (Cần map test_dataset về dạng ChatML giống train trước khi tính perplexity)
test_dataset_mapped = test_dataset.map(format_chatml)
eval_results = trainer.evaluate(eval_dataset=test_dataset_mapped)
eval_loss = eval_results['eval_loss']
perplexity = math.exp(eval_loss)

# Tổng hợp kết quả
final_metrics = {
    "ROUGE-1": rouge_results["rouge1"] * 100,
    "ROUGE-2": rouge_results["rouge2"] * 100,
    "ROUGE-L": rouge_results["rougeL"] * 100,
    "BLEU": bleu_results["score"], # SacreBLEU scale 0-100 sẵn
    "BERTScore F1": bert_f1 * 100,
    "Perplexity": perplexity
}

print("\n--- KẾT QUẢ ĐÁNH GIÁ ---")
for k, v in final_metrics.items():
    print(f"{k}: {v:.2f}")

Bắt đầu quá trình đánh giá trên tập Test...
